In [ ]:
!pip install datasets

In [2]:
!pip install faiss-cpu #required for RAG retriever

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 64.1 MB/s eta 0:00:00


In [7]:
from transformers import BertTokenizer,BertForQuestionAnswering,Trainer,TrainingArguments
from torch.utils.data import Dataset
import torch

In [10]:
#Sample task-specific corpus for QA
qa_corpus = [
    {
        "context": "Diabetes is a chronic condition characterized by high levels of sugar (glucose) in the blood. Common symptoms include increased thirst, frequent urination, and unexplained weight loss.",
        "question": "What are the symptoms of diabetes?",
        "answer": "Common symptoms include increased thirst, frequent urination, and unexplained weight loss."
    },
    {
        "context": "Hypertension, also known as high blood pressure, is a condition in which the force of the blood against the artery walls is too high. Often hypertension has no symptoms.",
        "question": "What is hypertension?",
        "answer": "Hypertension is a condition in which the force of the blood against the artery walls is too high."
    },
    {
        "context": "Asthma is a common long-term inflammatory disease of the airways of the lungs. Symptoms include episodes of wheezing, coughing, chest tightness, and shortness of breath.",
        "question": "What are the symptoms of asthma?",
        "answer": "Symptoms include episodes of wheezing, coughing, chest tightness, and shortness of breath."
    },
    {
        "context": "Coronary artery disease (CAD) is the most common type of heart disease. It occurs when the coronary arteries that supply blood to the heart muscle become hardened and narrowed due to the buildup of cholesterol and other material, called plaque, on their inner walls.",
        "question": "What is coronary artery disease?",
        "answer": "Coronary artery disease occurs when the coronary arteries that supply blood to the heart muscle become hardened and narrowed due to the buildup of cholesterol and other material, called plaque, on their inner walls."
    },
    {
        "context": "Osteoarthritis is the most common form of arthritis, affecting millions of people worldwide. It occurs when the protective cartilage that cushions the ends of the bones wears down over time. Symptoms include pain, stiffness, and loss of flexibility.",
        "question": "What are the symptoms of osteoarthritis?",
        "answer": "Symptoms include pain, stiffness, and loss of flexibility."
    },
    {
        "context": "Influenza, commonly known as the flu, is an infectious disease caused by an influenza virus. Symptoms can be mild to severe and commonly include a high fever, runny nose, sore throat, muscle and joint pain, headache, coughing, and feeling tired.",
        "question": "What are the symptoms of influenza?",
        "answer": "Symptoms commonly include a high fever, runny nose, sore throat, muscle and joint pain, headache, coughing, and feeling tired."
    },
    {
        "context": "Anemia is a condition in which you lack enough healthy red blood cells to carry adequate oxygen to your body's tissues. Having anemia can make you feel tired and weak. Symptoms can also include dizziness, shortness of breath, and pale skin.",
        "question": "What are the symptoms of anemia?",
        "answer": "Symptoms can include tiredness, weakness, dizziness, shortness of breath, and pale skin."
    },
    {
        "context": "Chronic obstructive pulmonary disease (COPD) is a chronic inflammatory lung disease that causes obstructed airflow from the lungs. Symptoms include breathing difficulty, cough, mucus (sputum) production, and wheezing.",
        "question": "What are the symptoms of COPD?",
        "answer": "Symptoms include breathing difficulty, cough, mucus (sputum) production, and wheezing."
    },
    {
        "context": "Gastroesophageal reflux disease (GERD) is a chronic digestive disease. GERD occurs when stomach acid or, occasionally, stomach content, flows back into your food pipe (esophagus). The backwash (acid reflux) irritates the lining of your esophagus and can cause heartburn.",
        "question": "What is GERD?",
        "answer": "GERD is a chronic digestive disease that occurs when stomach acid or stomach content flows back into your food pipe (esophagus). The backwash (acid reflux) irritates the lining of your esophagus and can cause heartburn."
    },
    {
        "context": "Chronic kidney disease (CKD) means your kidneys are damaged and can't filter blood the way they should. The disease is called 'chronic' because the damage to your kidneys happens slowly over a long period of time. Symptoms may include fatigue, swelling in your legs, ankles, or feet, and shortness of breath.",
        "question": "What are the symptoms of chronic kidney disease?",
        "answer": "Symptoms may include fatigue, swelling in your legs, ankles, or feet, and shortness of breath."
    }
]

In [11]:
# Custom dataset class for QA
class QADataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.examples = []
        for item in data:
            inputs = tokenizer.encode_plus(
                item['question'], item['context'],
                add_special_tokens=True,
                max_length=max_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            start_positions = inputs.input_ids.squeeze().tolist().index(tokenizer.encode(item['answer'], add_special_tokens=False)[0])
            end_positions = start_positions + len(tokenizer.encode(item['answer'], add_special_tokens=False)) - 1
            self.examples.append({
                'input_ids': inputs.input_ids.squeeze(),
                'attention_mask': inputs.attention_mask.squeeze(),
                'start_positions': torch.tensor(start_positions),
                'end_positions': torch.tensor(end_positions)
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]


In [12]:
# Load a pre-trained BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForQuestionAnswering.from_pretrained('bert-base-uncased')

# Prepare datasets
train_dataset = QADataset(qa_corpus, tokenizer)

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=10_000,
    save_total_limit=2,
)

# Create a trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

# Fine-tune the model
trainer.train()


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jas1420047 (jas1420047-mn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


TrainOutput(global_step=9, training_loss=4.526571061876085, metrics={'train_runtime': 365.0152, 'train_samples_per_second': 0.082, 'train_steps_per_second': 0.025, 'total_flos': 7838902702080.0, 'train_loss': 4.526571061876085, 'epoch': 3.0})

In [13]:
def generate_response_qa(question, context, model, tokenizer):
    inputs = tokenizer.encode_plus(question, context, return_tensors='pt')
    input_ids = inputs['input_ids'].tolist()[0]
    text_tokens = tokenizer.convert_ids_to_tokens(input_ids)

    outputs = model(**inputs)
    answer_start_scores = outputs.start_logits
    answer_end_scores = outputs.end_logits

    answer_start = torch.argmax(answer_start_scores)
    answer_end = torch.argmax(answer_end_scores) + 1

    answer = tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(input_ids[answer_start:answer_end]))
    return answer

# Sample question and context
context = "Diabetes is a chronic condition characterized by high levels of sugar (glucose) in the blood. Common symptoms include increased thirst, frequent urination, and unexplained weight loss."
question = "What are the symptoms of diabetes?"

response = generate_response_qa(question, context, model, tokenizer)
print(response)



symptoms of diabetes ? [SEP] diabetes is a chronic condition characterized by high levels of sugar ( glucose ) in the blood . common symptoms include increased thirst , frequent urination ,
